# 01 — House model: identity → pose set → clothing masks

**Phase-0, Stage A** of [03-generation-pipeline](../../docs/architecture/03-generation-pipeline.md).
Creates one synthetic house model in five fixed angles (front, 3/4 left, 3/4 right, side, back),
all in the same neutral outfit, light and backdrop. Then it computes clothing masks for each angle. Notebook `02` swaps garments onto these bases.

**Run on Kaggle (recommended)**: New Notebook → *File → Import Notebook* (this file) → Settings: *Accelerator = GPU T4*, *Internet = on* →
*Add-ons → Secrets*: `HF_TOKEN` (Hugging Face read token) and, if the repo is private, `GITHUB_TOKEN`.
**Colab**: Runtime → T4 GPU; add the same names under 🔑 Secrets.

You pick the best image at two points (identity, then one per angle). Everything is saved to `OUT_DIR` with a `meta.json`.

Models (see [licence register](../../docs/architecture/licence-register.md)):
FLUX.2 [klein] 4B (Apache 2.0) · SegFormer clothes parser (**research only until its licence is cleared**).

## 1. Config

In [ ]:
MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"   # official BFL org only (ADR-0004)
SEG_MODEL_ID = "mattmdjaga/segformer_b2_clothes"  # research only, see licence register

WIDTH, HEIGHT = 768, 1024        # portrait, multiples of 16
STEPS = 4                        # klein-4B is step-distilled; try 4–8
GUIDANCE = 1.0                   # ignored by distilled checkpoints (>1 only logs a warning)
FORCE_FP16 = False               # set True only on T4/P100 if bf16 is too slow and outputs look fine

IDENTITY_SEEDS = list(range(100, 108))   # 8 identity candidates
POSE_SEEDS = [1, 2, 3]                    # candidates per angle
ANGLES_TO_RENDER = ["front", "three_quarter_left", "three_quarter_right", "side", "back"]

import os
OUT_DIR = "/kaggle/working/outputs" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "outputs"

## 2. Setup

In [ ]:
# --- Get the repo helpers + install pinned deps (runs once per session) ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/vishwa9/tryon-app.git"
REPO_BRANCH = "main"   # use "research/klein-house-model" until that branch is merged

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/tryon-app"), Path("/content/tryon-app")]:
        if (p / "research" / "phase0_utils.py").exists():
            return p
    return None

REPO = _find_repo()
if REPO is None:
    REPO = Path("/kaggle/working/tryon-app" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content/tryon-app")
    url = REPO_URL
    # Private repo? Add a GITHUB_TOKEN secret (read-only, fine-grained) in Kaggle/Colab secrets.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
        except Exception:
            tok = os.environ.get("GITHUB_TOKEN")
    if tok:
        url = url.replace("https://", f"https://x-access-token:{tok}@")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, url, str(REPO)], check=True)
    if not (REPO / "research" / "phase0_utils.py").exists():
        raise SystemExit(f"Branch {REPO_BRANCH!r} has no research/phase0_utils.py. Set REPO_BRANCH to the branch with the notebooks.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "research" / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO / "research"))

import importlib, phase0_utils as u
importlib.reload(u)
print("repo:", REPO, "| runtime:", u.detect_runtime())

In [ ]:
# --- GPU check + Hugging Face login ---
import torch, diffusers
from huggingface_hub import login

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Kaggle: Settings → Accelerator → GPU T4. Colab: Runtime → Change runtime type → T4 GPU.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory/1e9:.1f} GB, compute {props.major}.{props.minor}")
print("torch", torch.__version__, "| diffusers", diffusers.__version__)

HF_TOKEN = u.get_secret("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("No HF_TOKEN secret found. Public models still download, but add one if you hit rate limits or gated repos.")

# T4/P100 (compute < 8.0) have no native bf16: it still works but is slower.
# float16 is faster on those cards but FLUX-family models can overflow to black/NaN images in fp16.
DTYPE = torch.bfloat16 if (props.major >= 8 or not FORCE_FP16) else torch.float16
print("dtype:", DTYPE)

def peak_vram_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 2)

## 3. Load FLUX.2 [klein] 4B

In [ ]:
from diffusers import Flux2KleinPipeline
timings = {}
with u.timer(timings, "load_s"):
    pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    pipe.enable_model_cpu_offload()   # fits a 16 GB card; costs some speed
print(timings, "| distilled:", pipe.config.get("is_distilled"))

def generate(prompt, seed, refs=None):
    torch.cuda.reset_peak_memory_stats()
    g = torch.Generator("cpu").manual_seed(seed)
    t = {}
    with u.timer(t, "s"):
        img = pipe(image=refs, prompt=prompt, width=WIDTH, height=HEIGHT,
                   num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=g).images[0]
    return img, t["s"], peak_vram_gb()

## 4. Identity

Edit the spec to taste. Keep the **outfit plain and fitted** (it's what gets masked and replaced),
and the **scene identical** for every angle. The model is synthetic, so never describe a real person.

In [ ]:
from pathlib import Path
spec = u.HouseModelSpec(
    name="Arjun",
    # person=..., outfit=..., scene=...   # override any field here
)
MODEL_DIR = Path(OUT_DIR) / "house_models" / spec.name
print(spec.identity_prompt())

identity_candidates, stats = [], []
for seed in IDENTITY_SEEDS:
    img, sec, vram = generate(spec.identity_prompt(), seed)
    identity_candidates.append(img); stats.append((seed, sec, vram))
    print(f"seed {seed}: {sec}s, peak {vram} GB")
u.make_grid(identity_candidates, [f"#{i} seed {s}" for i, (s, _, _) in enumerate(stats)], cols=4, cell=300)

In [ ]:
CHOSEN_IDENTITY = 0   # ← index from the grid above
identity = identity_candidates[CHOSEN_IDENTITY]
(MODEL_DIR / "identity").mkdir(parents=True, exist_ok=True)
identity.save(MODEL_DIR / "identity" / "0.png")
identity

## 5. Pose set

Each angle is generated with the chosen identity as a **reference image**, so the face and body carry over.
Look for: same face and hair, same outfit, same light and backdrop, full body in frame, natural hands.
The back view must not show the face.

In [ ]:
pose_candidates, pose_stats = {}, {}
for angle in ANGLES_TO_RENDER:
    pose_candidates[angle], pose_stats[angle] = [], []
    for seed in POSE_SEEDS:
        img, sec, vram = generate(spec.pose_prompt(angle), seed, refs=[identity])
        pose_candidates[angle].append(img); pose_stats[angle].append({"seed": seed, "sec": sec, "peak_vram_gb": vram})
    print(angle, [s["sec"] for s in pose_stats[angle]], "s")
    display(u.make_grid(pose_candidates[angle], [f"{angle} #{i}" for i in range(len(POSE_SEEDS))], cols=len(POSE_SEEDS), cell=300))

In [ ]:
CHOSEN_POSES = {a: 0 for a in ANGLES_TO_RENDER}   # ← e.g. {"front": 2, "side": 1, ...}
# Unhappy with an angle? Change POSE_SEEDS or tweak u.ANGLE_PHRASES[angle] and re-run the cell above for it.

bases = {}
for angle, idx in CHOSEN_POSES.items():
    d = MODEL_DIR / "poses" / angle
    d.mkdir(parents=True, exist_ok=True)
    bases[angle] = pose_candidates[angle][idx]
    bases[angle].save(d / "base.png")
u.make_grid(list(bases.values()), list(bases.keys()), cols=5, cell=260)

## 6. Clothing masks

A human-parsing model labels each pixel (upper clothes, pants, arms, face, ...). We save **hard** masks
per region. Notebook 02 grows and feathers them per garment. Check the overlays: the mask must cover the whole
t-shirt / trousers and **must not** cover the face, hair or hands.

In [ ]:
import numpy as np
from PIL import Image
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation

seg_proc = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model = AutoModelForSemanticSegmentation.from_pretrained(SEG_MODEL_ID).to("cuda").eval()

def parse(img):
    inputs = seg_proc(images=img, return_tensors="pt").to("cuda")
    with torch.no_grad():
        logits = seg_model(**inputs).logits
    up = torch.nn.functional.interpolate(logits, size=img.size[::-1], mode="bilinear", align_corners=False)
    return up.argmax(1)[0].cpu().numpy()

previews, mask_stats = [], {}
for angle, base in bases.items():
    lm = parse(base)
    d = MODEL_DIR / "poses" / angle
    np.save(d / "labels.npy", lm.astype(np.uint8))
    mask_stats[angle] = {}
    for region in ["upper", "lower", "full"]:
        m = u.region_mask(lm, region)
        u.mask_to_pil(m.astype(np.float32)).save(d / f"mask_{region}.png")
        mask_stats[angle][region] = round(float(m.mean()), 3)
    m_arms = u.region_mask(lm, "upper", include_arms=True)
    u.mask_to_pil(m_arms.astype(np.float32)).save(d / "mask_upper_arms.png")
    previews += [u.overlay_mask(base, u.region_mask(lm, "upper")), u.overlay_mask(base, u.region_mask(lm, "lower"))]
print(mask_stats)   # fraction of pixels per mask; ~0 means the parser missed the garment
u.make_grid(previews, [f"{a} {r}" for a in bases for r in ("upper", "lower")], cols=4, cell=260)

## 7. Save metadata (+ zip for notebook 02)

In [ ]:
import dataclasses, shutil, datetime
meta = {
    "name": spec.name,
    "spec": dataclasses.asdict(spec),
    "model_id": MODEL_ID, "seg_model_id": SEG_MODEL_ID,
    "diffusers": diffusers.__version__, "torch": torch.__version__, "dtype": str(DTYPE),
    "gpu": torch.cuda.get_device_name(0),
    "size": [WIDTH, HEIGHT], "steps": STEPS,
    "identity_seed": IDENTITY_SEEDS[CHOSEN_IDENTITY],
    "poses": {a: {**pose_stats[a][i], "chosen_index": i, "prompt": spec.pose_prompt(a)} for a, i in CHOSEN_POSES.items()},
    "mask_coverage": mask_stats,
    "load_s": timings.get("load_s"),
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
u.save_json(meta, MODEL_DIR / "meta.json")
archive = shutil.make_archive(str(Path(OUT_DIR) / f"house_model_{spec.name}"), "zip", MODEL_DIR)
print("saved", MODEL_DIR, "\nzip:", archive)
print("Kaggle: 'Save Version' → this output can be added as an input to notebook 02.")